<a href="https://colab.research.google.com/github/mmariannaholanda/ia_identificacao_de_digitos/blob/main/RECONHECIMENTO_DE_DIGITOS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# reconhecimento de digitos (MNIST) com PyTorch

vou construir a rede passo a passo que nem fiz nos tumores, mas agr com imagem e PyTorch
- primeiro o MLP (rede em camadas)
- depois a CNN com kernels, em outro notebook

primeiro os dados
MNIST não é uma tabela, é uma lista de pares (imagem, rótulo)
- 60000 imagens de treino e 10000 de teste, ja separadas
- train=True pega o treino e train=False pega o teste
- cada posição devolve 2 coisas: o tensor (imagem) e o rótulo (o dígito certo)

o que sao tensores?? computador não enxerga a imagem como nós enxergamos, ele representa cada pixel através de um número de 0 a 255
- o PyTorch trabalha com tensores pra fazer as operações matemáticas - tensor é uma estrutura usada pra armazenar e organizar números em uma ou mais dimensões

antes de fazer qualquer conta quero ver o que tem dentro
cada imagem original é uma grade 28x28 (uint8, de 0 a 255) e depois do ToTensor() vira (1, 28, 28), o 1 é o canal (tons de cinza)


In [1]:
# imports
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt  # biblioteca pra desenhar graficos e imagens (plt é so o apelido)

# datasets
from torchvision import datasets, transforms
# transforms = permite transformar imagens

dados_treino = datasets.MNIST(root="dados", train=True, download=True, transform=transforms.ToTensor())
dados_teste = datasets.MNIST(root="dados", train=False, download=True, transform=transforms.ToTensor())
# root="dados": pasta onde os arquivos do conjunto de dados serão armazenados
# train=True: indica que queremos os dados de treinamento
# train=False: indica que queremos os dados de teste
# transform=transforms.ToTensor(): converte as imagens para tensores do PyTorch


# --- olhando o que tem dentro ---
# cada imagem original tem shape (28, 28) e dtype=uint8: tipo numérico utilizado pra representar os pixels
# depois do ToTensor() o shape vira (1, 28, 28) e os valores viram float32 de 0 a 1
print(len(dados_treino), len(dados_teste))

imagem, rotulo = dados_treino[0]
# imagem = recebe os pixels da imagem; rotulo = recebe o número correto representado nela

print(imagem.shape, rotulo)

# desenhando a imagem crua (dados_treino.data guarda os pixels de 0 a 255)
plt.imshow(dados_treino.data[0], cmap="gray")
plt.title(f"rotulo: {dados_treino.targets[0].item()}")
plt.show()

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.28MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 155kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.45MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.1MB/s]


60000 10000
torch.Size([1, 28, 28]) 5


agora é a etapa de padronizar, que é diferente no contexto de imagem...
cada pixel vai de 0 a 255, nossas imagens tem 784 pixels
ToTensor() divide todo mundo por 255, ou seja, a escala fica de 0 a 1
mas só quando eu peço uma imagem (dados_treino[0]), o dados_treino.data guarda os pixels crus, entao ai eu divido por 255 na mão

media e desvio é 1 valor só pra todos os pixels de todas as imagens de treino
- cada imagem tem 784 pixels
- 60000 imagens de treino
- total de pixels: 784 × 60000 = 47.040.000
- soma todos os valores, divide por 47.040.000 e dá 0,1307

depois padroniza: cada pixel menos a média, dividido pelo desvio
o teste usa a média e o desvio do treino pra n vazar informação


São 60000 imagens de treino e 10000 de teste, MNIST não é uma tabela
MNIST é uma lista de pares de imagem, cada uma com uma posição, de 0 a 59999
1 posição devolve 2 coisas: tensor (imagem) e um rótulo (número)

agora deveria ser a etapa de padronizar, que é diferente no contexto de imagem...
cada pixel vai de 0 a 255, nossas imagens tem 784 pixels
ToTensor() pega e dividde todo mundo por 255, ou seja, agr a escala fica de 0 a 1
depois disso a gente padroniza - fNo MNIST, é a mesma conta
Cada imagem tem 784 pixels.
São 60000 imagens de treino.
Total de pixels: 784 × 60000 = 47.040.000.
Soma todos os valores, divide por 47.040.000, e dá 0,1307.

In [2]:
# pre processamento de dados em ML

# media e desvio de TODOS os pixels de TODAS as imagens de treino (1 valor só pra cada)
# divido por 255 pra ficar na escala de 0 a 1, a mesma q o ToTensor() entrega
media = dados_treino.data.float().mean() / 255
desvio = dados_treino.data.float().std() / 255

print(media, desvio)
# deve dar perto de 0.1307 e 0.3081


# standardization
# dados_treino.data são os pixels crus (0 a 255), entao divido por 255 antes de subtrair a media
treino_padronizado = (dados_treino.data.float() / 255 - media) / desvio
teste_padronizado = (dados_teste.data.float() / 255 - media) / desvio
# o teste usa a media e o desvio do treino

print(treino_padronizado.mean(), treino_padronizado.std())
# conferindo: deve dar perto de 0 e perto de 1

agora o baseline
nos tumores eram 2 classes, aqui são 10 (dígitos de 0 a 9) mas a ideia é a mesma: o "modelo" q sempre chuta a classe mais comum do treino
- conto quantas vezes cada dígito aparece nas respostas do treino (targets)
- o que aparece mais é a classe majoritária
- conto quantas vezes cada dígito aparece no teste
- baseline = vezes q a classe majoritária aparece no teste / 10000

In [3]:
# baseline
contagem_treino = torch.bincount(dados_treino.targets)  # conta quantas vezes cada digito (0 a 9) aparece
print(contagem_treino)

classe_majoritaria = contagem_treino.argmax()  # argmax = a posição do maior valor, q é o proprio digito
print(classe_majoritaria)

contagem_teste = torch.bincount(dados_teste.targets)

baseline = contagem_teste[classe_majoritaria] / len(dados_teste)
print(baseline)
# deve dar perto de 0.11


NameError: name 'torch' is not defined

agora entradas e respostas
antes treino_padronizado tinha formato (60000, 28, 28), ou seja, cada imagem era uma matriz 28x28
com reshape cada imagem vira uma fila de 784 números, que nem cada tumor era uma linha de 4 números
atencao: o pixel que estava logo acima de outro passa a ficar 28 posições longe dele na fila, a rede n sabe mais quem era vizinho de quem
é essa a limitação do MLP que a CNN resolve com os kernels, que deslizam pela grade sem achatar
as respostas ficam como estão

In [ ]:
# montando a base de ref da rede

# faço um reshape pq antes treino_padronizado tinha formato (60000, 28, 28), ou seja, cada imagem era uma matriz 28,28
# com reshape cada imagem virou uma fila de 784 numeros e fica melhor de trabalhar
entradas_treino = treino_padronizado.reshape(60000, 784)
respostas_treino = dados_treino.targets

entradas_teste = teste_padronizado.reshape(10000, 784)
respostas_teste = dados_teste.targets

print(entradas_treino.shape, respostas_treino.shape)  # (60000, 784) e (60000)
print(entradas_teste.shape, respostas_teste.shape)    # (10000, 784) e (10000)

agora a rede, peça por peça

camada escondida: cada neurônio faz a conta de sempre, soma de (pixel × peso) + viés
quero 128 neurônios olhando os 784 pixels, entao 784 entradas e 128 saídas
2 linhas, 2 momentos:
- criar (nn.Linear(784, 128)): aqui os pesos passam a existir, o PyTorch sorteia 784 × 128 pesos e 128 vieses e guarda dentro da camada, nenhuma imagem entrou ainda
- usar (camada_escondida(entradas_treino)): a camada faz a conta com os pesos q ja guarda
a semente (torch.manual_seed(42)) tem q vir antes de criar a camada, pq é na criação q o sorteio acontece
pra provar q ela faz a minha conta refaço na mão entradas @ pesos + vieses e comparo
o PyTorch guarda os pesos como (neurônios, entradas), ao contrario do NumPy dos tumores, entao uso .T (transposta) pra conta fechar

relu: a camada escondida devolve números q podem ser negativos, a relu deixa passar o positivo e troca o negativo por 0
sem ela as camadas empilhadas virariam uma conta só, é ela q dá a curva pra rede
a relu n tem pesos, só aplica a regra, entao o formato n muda: (60000, 128)

In [ ]:
# --- camada escondida (criar) ---
torch.manual_seed(42)  # fixa o sorteio dos pesos (tem q vir ANTES de criar as camadas)

camada_escondida = nn.Linear(784, 128)  # 784 entradas e 128 saídas
# essa parte é interessante pq o PyTorch sorteia 784 × 128 pesos aleatórios e 128 vieses
# e guarda dentro do objeto camada_escondida sem precisar de np.random.randn

print(camada_escondida.weight.shape)  # (128, 784): o PyTorch guarda (neuronios, entradas)
print(camada_escondida.bias.shape)    # (128,): 1 vies por neuronio


# --- camada escondida (usar) ---
saida_escondida = camada_escondida(entradas_treino)
print(saida_escondida.shape)  # (60000, 128): 60000 imagens, cada uma virou 128 numeros

# provando q a camada faz a minha conta
pesos = camada_escondida.weight
vieses = camada_escondida.bias

conta = entradas_treino @ pesos.T + vieses

print(torch.allclose(conta, saida_escondida))  # True


# --- relu ---
relu = nn.ReLU()

saida_relu = relu(saida_escondida)

print(saida_escondida[0][:5])  # antes da relu
print(saida_relu[0][:5])       # depois: os negativos viraram 0
print(saida_relu.shape)        # (60000, 128)

camada de saída
tenho saida_relu com formato (60000, 128)
quero 10 números por imagem, uma nota pra cada dígito (0 a 9), a maior nota é o palpite da rede
entao 128 entradas (os padrões da camada anterior) e 10 saídas
n coloco relu depois: as notas ficam cruas pq nota negativa tbm tem informação
quem transforma nota em probabilidade é o softmax, q entra na perda
os pesos ainda são aleatórios, entao essas 10 notas ainda n significam nada

agora junto a rede
tenho 3 peças soltas: camada_escondida, relu e camada_saida
o nn.Sequential junta as 3 na ordem certa e faz o papel da minha função propagar
ele n sorteia pesos, quem sorteou foram as camadas Linear na criação, o Sequential só guarda a ordem e a lista de pesos (modelo.parameters()) q o treino vai usar
total de parâmetros: 784 × 128 + 128 + 128 × 10 + 10 = 101.770

In [ ]:
# --- camada de saida ---
camada_saida = nn.Linear(128, 10)  # 128 entradas (os padrões) e 10 saídas (as notas)

saida_notas = camada_saida(saida_relu)

print(camada_saida.weight.shape)  # (10, 128)
print(saida_notas.shape)          # (60000, 10): 60000 imagens, 10 notas pra cada


# --- juntando tudo ---
# nn.Sequential faz o papel da sua função propagar
modelo = nn.Sequential(camada_escondida, relu, camada_saida)

saida_notas_modelo = modelo(entradas_treino)
print(saida_notas_modelo.shape)  # (60000, 10)
print(torch.allclose(saida_notas_modelo, saida_notas))  # True

total_parametros = sum(p.numel() for p in modelo.parameters())
print(total_parametros)  # 101770

agora a perda
as 10 notas são o fim da rede, n entram em outra camada, vão pra perda q compara com o dígito certo e devolve 1 número só: o quanto a rede errou
2 contas seguidas q o CrossEntropyLoss faz por dentro:
- softmax: transforma as 10 notas em probabilidades q somam 1
- entropia cruzada: perda = -ln(probabilidade do dígito certo), se a rede dá probabilidade alta pro certo a perda é pequena, se dá baixa a perda é grande
por isso as notas entram cruas, o softmax ja tá dentro
antes do treino a rede dá quase 1/10 pra cada dígito, entao a perda fica perto de -ln(0,1) = 2,3 (o equivalente ao 0,5550 do passo 0 dos tumores)

agora 1 passo de ajuste, antes de repetir milhares de vezes quero ver 1 passo funcionando
a ordem é sempre zero_grad() → backward() → step()
- zero_grad(): apaga os gradientes antigos, o PyTorch soma o gradiente novo ao q ja tá guardado, entao limpo pra n ficar somando
- backward(): calcula a derivada da perda em relação a cada peso e viés (backpropagation)
- step(): atualiza cada parâmetro com parametro - taxa × gradiente
se eu chamasse zero_grad() depois do backward() ele apagaria os gradientes antes do step() e nenhum peso mudaria
pro teste n mexer na rede original faço uma cópia (deepcopy)
a perda depois do passo tem q ser um pouco menor q a de antes

In [ ]:
# --- perda ---
funcao_perda = nn.CrossEntropyLoss()
perda = funcao_perda(saida_notas, respostas_treino)
print(perda.item())  # perto de 2.3


# --- 1 passo de ajuste ---
import copy

# faço uma COPIA da rede so pra testar 1 passo, assim a rede original continua intocada pro treino
modelo_teste = copy.deepcopy(modelo)

# taxa de aprendizado = 0.1
# optim: módulo que contém os algoritmos de otimização
# SGD: algoritmo Stochastic Gradient Descent (Descida do Gradiente Estocástico)
# modelo_teste.parameters(): obtém os parâmetros treináveis do modelo, ou seja, pesos e vieses
# otimizador = ATUALIZACAO DOS PARAMETROS
otimizador_teste = torch.optim.SGD(modelo_teste.parameters(), lr=0.1)

# calculamos o erro da rede neural
perda_antes = funcao_perda(modelo_teste(entradas_treino), respostas_treino)

otimizador_teste.zero_grad()  # apaga os gradientes antigos para n ficar somando
perda_antes.backward()        # calcula as derivadas parciais da função de perda em relação a cada peso e viés (backpropagation)
otimizador_teste.step()       # atualização dos parâmetros utilizando os gradientes calculados anteriormente
# zero_grad()  →  backward()  →  step()

perda_depois = funcao_perda(modelo_teste(entradas_treino), respostas_treino)
print(perda_antes.item(), perda_depois.item())  # perda_depois um pouco menor

agora o loop de treino
repito essas ações várias vezes na rede original, a cada passo:
- propaga e mede o erro com os pesos atuais
- guarda a perda na lista (historico_perda) pro gráfico
- ajusta: zero_grad() → backward() → step()

taxa_aprendizado é o tamanho do passinho e quantidade_passos é quantas vezes repito
cada passo usa as 60000 imagens de uma vez, igual nos tumores
n existe número certo de passos: rodo, olho a curva da perda e decido
começo com 100 pra ver rodando, depois aumento (tipo 5000) se a perda ainda cair bastante

In [ ]:
# loop
taxa_aprendizado = 0.1
quantidade_passos = 100  # comecando com 100 pra ver rodando, depois aumento

otimizador = torch.optim.SGD(modelo.parameters(), lr=taxa_aprendizado)
funcao_perda = nn.CrossEntropyLoss()

historico_perda = []  # lista vazia para guardar a perda de cada passo

for passo in range(quantidade_passos):

    # 1) propagar e medir o erro com os pesos atuais
    notas = modelo(entradas_treino)
    perda = funcao_perda(notas, respostas_treino)
    historico_perda.append(perda.item())

    # 2) ajustar: limpar, calcular gradientes, andar
    otimizador.zero_grad()
    perda.backward()
    otimizador.step()

    if passo % 10 == 0:
        print(f"passo {passo}: perda = {perda.item():.4f}")

print(f"perda final: {historico_perda[-1]:.4f}")

agora o gráfico da perda
se a curva cai e depois achata o treino funcionou e mais passos ajudam pouco
se ainda cai bastante no fim eu aumento a quantidade_passos

curiosidade: cada neurônio da camada escondida tem 784 pesos, 1 por pixel, dá pra reorganizar em 28x28 e desenhar
antes do treino parecia ruído de TV pq os pesos eram aleatórios, agr depois do treino comparo

In [ ]:
# gráfico da perda
plt.figure(figsize=(8, 5))
plt.plot(historico_perda)
plt.xlabel("Passo")
plt.ylabel("Perda (entropia cruzada)")
plt.title("Perda do treino a cada passo")
plt.grid(True)

plt.savefig("perda_treino.png", dpi=150, bbox_inches="tight")

plt.show()


# --- pesos de 1 neuronio (depois do treino) ---
pesos_neuronio = camada_escondida.weight[0].detach().reshape(28, 28)
# weight[0] = os 784 pesos do primeiro neuronio
# detach() = so quero olhar os numeros, sem gradiente
plt.imshow(pesos_neuronio, cmap="gray")
plt.show()

agora a avaliação, começando pela acurácia
a rede devolve 10 notas por imagem e a decisão é o dígito de maior nota
argmax(dim=1) devolve a posição da maior nota de cada imagem (o dim=1 é "procura em cada linha", ou seja, em cada imagem)
depois comparo com a resposta certa: previsoes == respostas dá True/False, o .float() vira 1.0/0.0 e o .mean() dá a fração de acertos
torch.no_grad() desliga o cálculo de gradientes pq aqui só tô medindo
faço no treino e no teste (dados q a rede nunca viu) e comparo com o baseline

agora o recall por dígito
recall responde sobre 1 classe: das imagens q eram esse dígito quantas a rede achou, com 10 classes tem 10 recalls
pra cada dígito "positivo" é ser aquele dígito e "negativo" é ser qualquer um dos outros 9:
- verdadeiros positivos: era o dígito e a rede disse o dígito
- falsos negativos: era o dígito e a rede disse outro
- falsos positivos: n era o dígito e a rede disse o dígito
- verdadeiros negativos: n era o dígito e a rede disse outro
nos tumores == 0 era "benigno", aqui == 0 seria "o dígito 0", por isso uso != digito pra "qualquer outro"
a soma das quatro tem q dar 10000 em todos os dígitos (é o meu teste do 570)
guardo os recalls numa lista e tiro a média no final

In [ ]:
# --- acuracia ---

# torch.no_grad() é utilizado para desativar o cálculo e o armazenamento dos gradientes durante determinado trecho do código
with torch.no_grad():  # so estamos medindo, sem calcular gradientes
    notas_treino = modelo(entradas_treino)
    previsoes_treino = notas_treino.argmax(dim=1)
    acuracia_treino = (previsoes_treino == respostas_treino).float().mean().item()

    # no teste (dados que a rede nunca viu)
    notas_teste = modelo(entradas_teste)
    previsoes_teste = notas_teste.argmax(dim=1)
    acuracia_teste = (previsoes_teste == respostas_teste).float().mean().item()


acuracia_treino_percentual = acuracia_treino * 100
acuracia_teste_percentual = acuracia_teste * 100
baseline_percentual = baseline.item() * 100

print(f"Acurácia no treino: {acuracia_treino_percentual:.3f} %")
print(f"Acurácia no teste:  {acuracia_teste_percentual:.3f} %")
print(f"Baseline (sempre o digito mais comum): {baseline_percentual:.3f} %")
print(f"Ganho sobre o baseline: {acuracia_teste_percentual - baseline_percentual:.3f} pontos")


# --- aqui eu faço o recall: ---

recalls = []

for digito in range(10):
    verdadeiros_positivos = ((previsoes_teste == digito) & (respostas_teste == digito)).sum().item()
    falsos_negativos = ((previsoes_teste != digito) & (respostas_teste == digito)).sum().item()
    falsos_positivos = ((previsoes_teste == digito) & (respostas_teste != digito)).sum().item()
    verdadeiros_negativos = ((previsoes_teste != digito) & (respostas_teste != digito)).sum().item()

    recall_digito = verdadeiros_positivos / (verdadeiros_positivos + falsos_negativos)
    recall_digito_percentual = recall_digito * 100
    recalls.append(recall_digito)

    print(f"===== Dígito {digito} =====")
    print(f"Verdadeiros positivos: {verdadeiros_positivos}")
    print(f"Falsos negativos: {falsos_negativos}")
    print(f"Falsos positivos: {falsos_positivos}")
    print(f"Verdadeiros negativos: {verdadeiros_negativos}")
    print(f"Soma das quatro: {verdadeiros_positivos + falsos_negativos + falsos_positivos + verdadeiros_negativos}")
    print(f"Recall: {recall_digito_percentual:.3f} %\n")

recall_medio = sum(recalls) / len(recalls)
print(f"Recall médio dos 10 dígitos: {recall_medio * 100:.3f} %")

agora a matriz de confusão
n são 10 matrizes, é 1 só de 10x10 (100 combinações)
- linha = dígito real
- coluna = dígito q a rede previu
- cada célula = quantas imagens do teste caíram naquela combinação

a diagonal tem os acertos, tudo fora é erro e a posição diz qual: a célula [4, 9] é "era 4 e a rede disse 9"
o zip pega 2 listas e anda com elas juntas, entregando 1 par por volta, que nem um zíper
cada volta é 1 imagem: pego o par (real, previsto) e somo 1 na célula [real, previsto]
a soma da matriz tem q dar 10000

depois desenho no mesmo estilo da dos tumores, com o número dentro de cada quadrado
os acertos formam a diagonal mais escura e os erros ficam em cor clara, mas os números continuam lá
procuro as células fora da diagonal com os maiores números, são os pares q a rede mais confunde

In [ ]:
# --- construindo a matriz ---
matriz_confusao = torch.zeros(10, 10, dtype=torch.int64)
# 10 linhas (real) x 10 colunas (previsto), tudo zero e so numeros inteiros

for digito_real, digito_previsto in zip(respostas_teste, previsoes_teste):  # pega 2 listas e anda com elas juntas, entregando um par por volta, como um zíper
    matriz_confusao[digito_real, digito_previsto] += 1
# para cada imagem do teste, soma 1 na celula [real, previsto]

print(matriz_confusao)
print("Soma da matriz:", matriz_confusao.sum().item())  # 10000


# --- agora eu desenho ---
plt.figure(figsize=(8, 7))
plt.imshow(matriz_confusao, cmap="Blues")

nomes_classes = [str(d) for d in range(10)]
plt.xticks(range(10), nomes_classes)
plt.yticks(range(10), nomes_classes)
plt.xlabel("Previsto")
plt.ylabel("Real")
plt.title("Matriz de confusão - Dígitos")

for linha in range(10):
    for coluna in range(10):
        valor = matriz_confusao[linha, coluna].item()
        cor_texto = "white" if valor > matriz_confusao.max().item() / 2 else "black"
        plt.text(coluna, linha, str(valor), ha="center", va="center", color=cor_texto, fontsize=9)

plt.savefig("matriz_confusao_digitos.png", dpi=150, bbox_inches="tight")
plt.show()

agora prever imagens

primeiro 1 imagem do teste, equivalente ao prever_tumor
- passo a fatia entradas_teste[posicao:posicao+1] pra continuar com formato (1, 784)
- o softmax transforma as 10 notas em probabilidades e a probabilidade do palpite é a confiança da rede
- a confiança diz o quanto a rede tá segura mas n prova q acertou
- desenho a imagem crua (dados_teste.data) mas a rede recebe a padronizada (entradas_teste)
troco posicao por vários números (0 a 9999) e procuro 1 imagem q a rede acerta e 1 q erra

depois a minha imagem
a rede só conhece imagem tipo MNIST, entao a minha precisa virar a mesma coisa antes de entrar:
- tons de cinza (.convert("L")): 1 número por pixel
- 28x28 (.resize)
- traço branco em fundo preto: no MNIST o fundo é preto (perto de 0) e o traço é branco (perto de 255), foto de caneta preta em papel branco é o contrário, entao preciso inverter
- se a média dos pixels passa de 127 o fundo é claro e inverto com 255 - pixels
- mesma padronização do treino: divido por 255 e uso a media e o desvio do treino
- fila de 784 números (reshape(1, 784))

desenho o q a rede recebeu pra conferir: se eu mesma n reconheço o dígito ali, o problema tá na preparação da imagem e n na rede
dica: traço grosso, imagem quadrada e dígito centralizado
a rede costuma ir pior nas minhas imagens do q no teste do MNIST pq traço, posição e iluminação são diferentes

In [ ]:
# --- 1 imagem do teste ---
posicao = 0  # qual imagem do teste olhar

with torch.no_grad():
    notas = modelo(entradas_teste[posicao:posicao+1])  # fatia, pra continuar sendo (1, 784)
    probabilidades = torch.softmax(notas, dim=1)

palpite = notas.argmax(dim=1).item()
confianca = probabilidades[0, palpite].item() * 100
resposta_certa = respostas_teste[posicao].item()

print(f"Resposta certa: {resposta_certa}")
print(f"Palpite da rede: {palpite} (confiança {confianca:.1f} %)")

plt.imshow(dados_teste.data[posicao], cmap="gray")
plt.title(f"Rede previu {palpite}, resposta certa {resposta_certa}")
plt.show()


# --- a minha imagem ---
from google.colab import files
from PIL import Image

arquivo_enviado = files.upload()  # abre o botao pra escolher a imagem
nome_arquivo = list(arquivo_enviado.keys())[0]

imagem_original = Image.open(nome_arquivo).convert("L")  # "L" = tons de cinza
imagem_reduzida = imagem_original.resize((28, 28))
pixels = torch.tensor(np.array(imagem_reduzida), dtype=torch.float32)  # 28x28, de 0 a 255

if pixels.mean() > 127:  # fundo claro: inverte pra ficar como o MNIST
    pixels = 255 - pixels

entrada_nova = (((pixels / 255) - media) / desvio).reshape(1, 784)  # mesma padronizacao do treino (tensor de 784 números)

with torch.no_grad():
    notas = modelo(entrada_nova)
    probabilidades = torch.softmax(notas, dim=1)

palpite = notas.argmax(dim=1).item()
confianca = probabilidades[0, palpite].item() * 100

print(f"Palpite da rede: {palpite} (confiança {confianca:.1f} %)")

plt.imshow(pixels, cmap="gray")
plt.title(f"O que a rede viu. Palpite: {palpite}")
plt.show()

agora salvo o modelo
salvo pesos, vieses, média e desvio num .npz pra prever depois sem treinar de novo
- .detach().numpy() tira os números do PyTorch (o detach é "só quero os números, sem gradiente")
- modelo[0] é a camada_escondida, modelo[1] é a relu (sem pesos) e modelo[2] é a camada_saida
- media e desvio agr são tensores de 1 número só, entao uso .item(), n são do pandas que nem nos tumores

atencao: ao carregar uso outro nome (arquivo_modelo), se eu escrevesse modelo = np.load(...) apagaria a minha rede
os pesos ficam transpostos em relação ao arquivo dos tumores: pesos_escondida é (128, 784) e n (784, 128)
essa célula só roda depois do treino, senão salvo os pesos aleatórios

In [ ]:
# implementando np.savez
# media e desvio agr são tensores de 1 número só (n são mais do pandas)


pesos_escondida = modelo[0].weight.detach().numpy()
vies_escondida = modelo[0].bias.detach().numpy()
pesos_saida = modelo[2].weight.detach().numpy()
vies_saida = modelo[2].bias.detach().numpy()
print("\n========================================================================\n") # organizando



np.savez(
    "reconhecimento_digitos.npz",  # função do NumPy que salva vários arrays em um único arquivo .npz
    pesos_escondida=pesos_escondida,
    vies_escondida=vies_escondida,
    pesos_saida=pesos_saida,
    vies_saida=vies_saida,
    media=media.item(),  # extrair o valor numérico de um tensor
    desvio=desvio.item(),
)
print("modelo salvo ok\n")
print("conferindo...")

arquivo_modelo = np.load("reconhecimento_digitos.npz")  # outro nome pra n apagar a rede (modelo)
for nome in arquivo_modelo.files:
    print(nome, arquivo_modelo[nome].shape)

    # vai me dizer linhas e colunas
    # media e desvio são 1 número só (shape ()), os vieses são vetores e os pesos são matrizes


# dando uma olhada em como ficou cada formato visualmente
print("\n=========================================================================\n")
np.set_printoptions(precision=3, suppress=True, linewidth=150)

for nome in arquivo_modelo.files:
    print("=====", nome, arquivo_modelo[nome].shape)
    print(arquivo_modelo[nome])